### Realizando junção e processamento das bases de dados iniciais

In [1]:
# Importando biblioteca necessária
import pandas as pd
import re

#### 1. Junção das bases de dados

In [2]:
# Função para padronizar dataset enron
def padroniza_enron(caminho):
    df = pd.read_csv(caminho)
    df_padrao = pd.DataFrame({
        'subject': df['subject'],
        'body': df['body'],
        'fonte': 'enron',
        'is_malicious': df['label']
    })
    return df_padrao

# Função para padronizar dataset nazario
def padroniza_nazario(caminho):
    df = pd.read_csv(caminho)
    df_padrao = pd.DataFrame({
        'subject': df['subject'],
        'body': df['body'],
        'fonte': 'nazario',
        'is_malicious': df['label']
    })
    return df_padrao

# Função para padronizar dataset spamAssasin
def padroniza_spamAssasin(caminho):
    df = pd.read_csv(caminho)
    df_padrao = pd.DataFrame({
        'subject': df['subject'],
        'body': df['body'],
        'fonte': 'spamAssasin',
        'is_malicious': df['label']
    })
    return df_padrao

A variável alvo foi definida como *is_malicious*, sendo composta por e-mails de *spam* provenientes das fontes *Enron* e *SpamAssassin*, além de e-mails de *phishing* provenientes da fonte *Nazario*.

In [3]:
# Chamando funções para padronização dos datasets
df_enron = padroniza_enron('../data/raw/Enron.csv')
df_nazario = padroniza_nazario('../data/raw/Nazario.csv')
df_spamAssasin = padroniza_spamAssasin('../data/raw/SpamAssasin.csv')

# Concatenando datasets
df_completo = pd.concat([df_enron, df_nazario, df_spamAssasin], ignore_index = True)

#### 2. Processamento da base de dados inicial

##### 2.1. Contagem de *urls* e ajuste na variável *urls*

In [4]:
# Definindo padrao url para aceitar http://, www. e forma tokenizada do Enron (http : / / ...)
padrao_url = re.compile(r'(?:https?\s*:\s*/\s*/|www\s*\.)', re.IGNORECASE)

# Conta numero de urls em cada body
df_completo['num_urls'] = df_completo['body'].str.count(padrao_url)

# Indicador binário de urls
df_completo['urls'] = (df_completo['num_urls'] > 0).astype(int)

# Percentual de e-mails que contém urls por fonte e por classe
df_completo.groupby(['fonte', 'is_malicious'])['urls'].mean().unstack().round(3)

is_malicious,0,1
fonte,,
enron,0.095,0.339
nazario,NaN,0.165
spamAssasin,0.872,0.607


##### 2.2. Normalização dos textos de *body* e *subject*

In [5]:
# Indicador de subject ausente
df_completo['subject_vazio'] = df_completo['subject'].isnull().astype(int)

# Função para normalizar o texto
def normaliza(serie):
    return (serie
            # Substitui qualquer sequencia de espaços em branco(tabs, espaços múltiplos, \n \r) 
            .str.replace(r'\s+', ' ', regex=True)
            # Remove espaços em branco no início em fim da string
            .str.strip()
            # Transforma todos os caracteres para minúsculo
            .str.lower())

# Chamando função para normalização de subject e body
df_completo['subject'] = normaliza(df_completo['subject'].fillna(''))
df_completo['body'] = normaliza(df_completo['body'])

##### 2.3. Remoção de *body* nulos e duplicatas

In [6]:
# Removendo emails com valores nulos para body
df_completo = df_completo.dropna(subset=['body'])

# Removendo duplicatas
df_completo = df_completo.drop_duplicates(subset=['subject', 'body'])

##### 2.4. Remoção de quase duplicatas

In [7]:
# Chave: apenas letras, primeiros 300 caracteres do body normalizado
df_completo['chave'] = (df_completo['body']
                        # Remove tudo que não seja letras minusculas, mantendo apenas uma sequencia de caracteres
                        .str.replace(r'[^a-z]', '', regex=True)
                        # Pega apenas os primeiros 300 caracteres
                        .str[:300])

# Uma chave é valida quando seu tamanho é maior ou igual a 30
chave_valida = df_completo['chave'].str.len() >= 30
# Verificando duplicatas e apenas selecionando aquelas que tem tamanho maior igual a 30, mantendo primeira ocorrência
duplicada = chave_valida & df_completo.duplicated('chave', keep='first')

print(f'Quase-duplicatas a remover: {duplicada.sum()}')
# Origem e classe das duplicatas
print(df_completo[duplicada].groupby(['fonte', 'is_malicious']).size().unstack())

# Verificação de inconsistência nas chaves
grupos = df_completo[chave_valida].groupby('chave')['is_malicious'].nunique()
print(f'Grupos com labels conflitantes: {(grupos > 1).sum()}')

# Removendo duplicatas e coluna auxiliar chave
df_completo = df_completo[~duplicada].drop(columns='chave')

Quase-duplicatas a remover: 3128
is_malicious       0      1
fonte                      
enron         1210.0  964.0
nazario          NaN  121.0
spamAssasin     97.0  736.0
Grupos com labels conflitantes: 0


Como a remoção de quase duplicatas ocorre enquanto as bases de dados ainda estão em ordem sequencial, a primeira ocorrência mantida tende a ser da fonte Enron, por ser a primeira concatenada.

##### 2.5. Remoção de *body* curtos

In [8]:
# Removendo bodys curtos
tamanho_minimo = 10
# Criando serie booleana indicando as instancias com body menor ou igual a 10
curtos = df_completo['body'].str.len() <= tamanho_minimo
print(f'Body curtos a remover: {curtos.sum()}')

# Removendo bodys curtos
df_completo = df_completo[~curtos].reset_index(drop=True)

Body curtos a remover: 57


##### 2.6. Remoção de registros corrompidos

Durante a análise exploratória foram identificados 2 registros da fonte Nazario cujo body não corresponde a um e-mail real, e sim a um artefato de extração (*placeholder*): o texto padrão que arquivos mbox inserem como primeira entrada ("*this text is part of the internal format of your mail folder, and is not a real message...*"). Um deles tem 4,5 milhões de caracteres, o *placeholder* seguido de um despejo de dados não relacionado, e o outro é só o *placeholder*, rotulado como malicioso sem conter conteúdo de *phishing* algum. Como não são e-mails de fato, ambos foram removidos abaixo.

In [9]:
# Identificando registros cujo body é o placeholder , não um e-mail real
placeholder = df_completo['body'].str.contains('is not a real message', regex=False)

print(f'Registros corrompidos a remover: {placeholder.sum()}')
print(df_completo[placeholder][['fonte', 'is_malicious']])

# Removendo registros da base de dados
df_completo = df_completo[~placeholder].reset_index(drop=True)

Registros corrompidos a remover: 2
         fonte  is_malicious
27201  nazario             1
27262  nazario             1


O processamento inicial da base envolveu, primeiramente, o recálculo da variável *url*, corrigindo inconsistências identificadas nos dados originais e obtendo também a quantidade de URLs por e-mail.

Em seguida, o texto de *subject* e *body* foi normalizado. Após a normalização, foram removidos e-mails com *body* nulo e registros duplicados, além de e-mails quase duplicados, identificados a partir de chaves de 300 caracteres alfabéticos. Registros cuja chave fosse idêntica e tivesse ao menos 30 caracteres de comprimento foram considerados quase duplicados e removidos. 

Por fim, foram removidos e-mails cujo *body* possuía 10 caracteres ou menos.

##### 2.7. Embaralhando, validando e salvando

In [10]:
# Embaralhando os dados para evitar que a concatenação por blocos deixe o treino enviesado
df_completo = df_completo.sample(frac=1, random_state=42).reset_index(drop=True)

# Validações antes de salvar
assert df_completo.isnull().sum().sum() == 0
assert not df_completo.duplicated(['subject', 'body']).any()
assert set(df_completo['is_malicious'].unique()) == {0, 1}

# Escrevendo o DataFrame final em CSV
df_completo.to_csv('../data/processed/phishing-dataset-merged.csv', index=False)